In [1]:
"""
02_create_maps.py
──────────────────
Generates interactive HTML maps from the processed incident data.
Open any output file in a browser — no server required.

MAPS PRODUCED:
  Per-year incident maps  →  output/maps/orlando_YYYY_incidents.html
  Per-year heat maps      →  output/maps/orlando_YYYY_heatmap.html
  All-years heat map      →  output/maps/orlando_all_years_heatmap.html
  All-years combined dot  →  output/maps/orlando_all_years_combined.html

TWEAK IDEAS:
  - CLUSTER_MARKERS: False gives raw dots, True groups them at low zoom
  - WEIGHT_BY_SEVERITY: counts fatality incidents more heavily on heat maps
  - Change HEAT_RADIUS / HEAT_BLUR in config.py to adjust heat map spread
"""

import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import branca.colormap as cm

import config

# ── Settings ───────────────────────────────────────────────────────────────────
CLUSTER_MARKERS    = True    # Cluster dots at low zoom (cleaner for dense areas)
WEIGHT_BY_SEVERITY = True    # Fatality incidents weighted heavier on heat maps
SHOW_MASS_SHOOTINGS = True   # Add a separate layer highlighting mass shootings

In [2]:
# ── Data ───────────────────────────────────────────────────────────────────────

def load_data() -> pd.DataFrame:
    path = config.DATA_PROCESSED / "orlando_incidents.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Processed data not found at {path}\n"
            "Run 01_prepare_data.py first."
        )
    df = pd.read_csv(path, parse_dates=["date"])
    df["year"] = df["date"].dt.year
    # Keep only rows with valid coordinates
    df = df.dropna(subset=["lat", "lon"])
    print(f"Loaded {len(df):,} mappable incidents.")
    return df


In [3]:
# ── Helpers ────────────────────────────────────────────────────────────────────

def _marker_color(row) -> str:
    if row["killed"] >= 4 or row["total"] >= 4:
        return "darkred"    # Mass shooting
    if row["killed"] > 0:
        return "red"        # Fatal
    if row["injured"] >= 3:
        return "orange"     # Multiple injured
    return "cadetblue"      # Injury only


def _popup(row) -> str:
    date_str = row["date"].strftime("%B %d, %Y") if pd.notna(row["date"]) else "Unknown"
    return (
        f"<b>{date_str}</b><br>"
        f"{row.get('address', '')}<br>"
        f"<b>Killed:</b> {row['killed']} &nbsp; <b>Injured:</b> {row['injured']}"
        + (" &nbsp; <b>⚠ Mass Shooting</b>" if row.get("mass") else "")
    )


def _heat_weight(row) -> float:
    if not WEIGHT_BY_SEVERITY:
        return 1.0
    return float(1 + row["killed"] * (config.FATALITY_WEIGHT - 1))


def _legend(colors: dict) -> folium.Element:
    items = "".join(
        f'<div><span style="color:{c};font-size:16px;">●</span> {label}</div>'
        for label, c in colors.items()
    )
    html = f"""
    <div style="position:fixed;bottom:30px;left:30px;background:white;
         padding:10px 14px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:13px;">
        <b>Incident Type</b><br>{items}
    </div>"""
    return folium.Element(html)


def _base_map() -> folium.Map:
    return folium.Map(
        location=[config.ORLANDO_LAT, config.ORLANDO_LON],
        zoom_start=config.DEFAULT_ZOOM,
        tiles="CartoDB positron",
    )

In [4]:
# ── Per-year incident dot map ──────────────────────────────────────────────────

def map_by_year(df: pd.DataFrame, year: int) -> None:
    year_df = df[df["year"] == year]
    if year_df.empty:
        print(f"  {year}: no data — skipping.")
        return

    m = _base_map()
    container = MarkerCluster(name="Incidents") if CLUSTER_MARKERS else folium.FeatureGroup(name="Incidents")

    for _, row in year_df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            color=_marker_color(row),
            fill=True,
            fill_opacity=0.75,
            popup=folium.Popup(_popup(row), max_width=300),
        ).add_to(container)

    container.add_to(m)

    if SHOW_MASS_SHOOTINGS:
        mass_df = year_df[year_df["mass"] == True]
        if not mass_df.empty:
            mass_layer = folium.FeatureGroup(name="Mass Shootings (4+ victims)")
            for _, row in mass_df.iterrows():
                folium.Marker(
                    location=[row["lat"], row["lon"]],
                    icon=folium.Icon(color="black", icon="warning-sign", prefix="glyphicon"),
                    popup=folium.Popup(_popup(row), max_width=300),
                ).add_to(mass_layer)
            mass_layer.add_to(m)

    folium.LayerControl().add_to(m)
    m.get_root().html.add_child(_legend({
        "Fatal (no mass)": "red",
        "Multiple injured": "orange",
        "Injury only": "cadetblue",
        "Mass shooting": "darkred",
    }))

    count = len(year_df)
    killed = year_df["killed"].sum()
    injured = year_df["injured"].sum()
    title_html = f"""
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
         background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:14px;text-align:center;">
        <b>Orlando Gun Violence {year}</b><br>
        {count:,} incidents &nbsp;|&nbsp; {killed:,} killed &nbsp;|&nbsp; {injured:,} injured
    </div>"""
    m.get_root().html.add_child(folium.Element(title_html))

    out = config.OUTPUT_MAPS / f"orlando_{year}_incidents.html"
    m.save(str(out))
    print(f"  Saved: {out.name}")


# ── Per-year heat map ──────────────────────────────────────────────────────────

def heatmap_by_year(df: pd.DataFrame, year: int) -> None:
    year_df = df[df["year"] == year]
    if year_df.empty:
        return

    m = _base_map()
    heat_data = [[row["lat"], row["lon"], _heat_weight(row)] for _, row in year_df.iterrows()]

    HeatMap(
        heat_data,
        name="Heat Map",
        radius=config.HEAT_RADIUS,
        blur=config.HEAT_BLUR,
        min_opacity=config.HEAT_MIN_OPACITY,
    ).add_to(m)

    title_html = f"""
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
         background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:14px;text-align:center;">
        <b>Orlando Gun Violence Heat Map — {year}</b>
        {"<br><small>Weighted by fatalities</small>" if WEIGHT_BY_SEVERITY else ""}
    </div>"""
    m.get_root().html.add_child(folium.Element(title_html))

    out = config.OUTPUT_MAPS / f"orlando_{year}_heatmap.html"
    m.save(str(out))
    print(f"  Saved: {out.name}")

In [5]:
# ── All-years heat map (layer per year, toggleable) ────────────────────────────

def heatmap_all_years(df: pd.DataFrame) -> None:
    m = _base_map()
    years = sorted(df["year"].unique())

    for i, year in enumerate(years):
        year_df = df[df["year"] == year]
        heat_data = [[row["lat"], row["lon"], _heat_weight(row)] for _, row in year_df.iterrows()]
        # Show only the most recent year by default
        HeatMap(
            heat_data,
            name=str(year),
            show=(i == len(years) - 1),
            radius=config.HEAT_RADIUS,
            blur=config.HEAT_BLUR,
            min_opacity=config.HEAT_MIN_OPACITY,
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    title_html = """
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
         background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:14px;text-align:center;">
        <b>Orlando Gun Violence — Heat Maps by Year</b><br>
        <small>Use layer control (top right) to switch years</small>
    </div>"""
    m.get_root().html.add_child(folium.Element(title_html))

    out = config.OUTPUT_MAPS / "orlando_all_years_heatmap.html"
    m.save(str(out))
    print(f"  Saved: {out.name}")

In [6]:
# ── All-years combined dot map (layer per year) ────────────────────────────────

def map_all_years_combined(df: pd.DataFrame) -> None:
    m = _base_map()
    years = sorted(df["year"].unique())

    year_colormap = cm.linear.YlOrRd_09.scale(min(years), max(years))
    year_colormap.caption = "Year"

    for i, year in enumerate(years):
        year_df = df[df["year"] == year]
        group = folium.FeatureGroup(name=str(year), show=(i == len(years) - 1))
        color = year_colormap(year)

        for _, row in year_df.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=5,
                color=color,
                fill=True,
                fill_opacity=0.65,
                popup=folium.Popup(_popup(row), max_width=280),
            ).add_to(group)

        group.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    year_colormap.add_to(m)

    title_html = """
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
         background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:14px;text-align:center;">
        <b>Orlando Gun Violence — All Years</b><br>
        <small>Toggle years using the layer control (top right)</small>
    </div>"""
    m.get_root().html.add_child(folium.Element(title_html))

    out = config.OUTPUT_MAPS / "orlando_all_years_combined.html"
    m.save(str(out))
    print(f"  Saved: {out.name}")


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    config.OUTPUT_MAPS.mkdir(parents=True, exist_ok=True)
    df = load_data()
    years = sorted(df["year"].unique())

    print(f"\nGenerating per-year maps for: {years}")
    for year in years:
        map_by_year(df, year)
        heatmap_by_year(df, year)

    print("\nGenerating multi-year maps...")
    heatmap_all_years(df)
    map_all_years_combined(df)

    print(f"\nDone. Open files in output/maps/ in any browser.")


if __name__ == "__main__":
    main()

Loaded 1,742 mappable incidents.

Generating per-year maps for: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
  Saved: orlando_2014_incidents.html
  Saved: orlando_2014_heatmap.html
  Saved: orlando_2015_incidents.html
  Saved: orlando_2015_heatmap.html
  Saved: orlando_2016_incidents.html
  Saved: orlando_2016_heatmap.html
  Saved: orlando_2017_incidents.html
  Saved: orlando_2017_heatmap.html
  Saved: orlando_2018_incidents.html
  Saved: orlando_2018_heatmap.html
  Saved: orlando_2019_incidents.html
  Saved: orlando_2019_heatmap.html
  Saved: orlando_2020_incidents.html
  Saved: orlando_2020_heatmap.html
  Saved: orlando_2021_incidents.html
  Saved: orlando_2021_heatmap.html
  Saved: orlando_2022_incidents.html
  Saved: orlando_2022_heatmap.html
  Saved: orlando_2023_incidents.html
  Saved: orlando_2023_heatmap.html
  Saved

In [7]:
# Interactive all-years map (zoom-adaptive: clusters -> choropleth -> dots)
#
# Zoom < 11   : MarkerCluster (incident-type colors)
# Zoom 11-13  : Neighborhood choropleth (density for selected years)
# Zoom >= 14  : Individual dots (incident-type colors, clickable)
# Right panel : year checkboxes with All/None buttons
# Layer ctrl  : All Neighborhoods | Preferred Neighborhoods | Kidz Zones
# Output      : output/maps/orlando_interactive_full.html

import json as _json
import geopandas as gpd

PREFERRED_NAMES = ["Holden/Parramore", "Holden Heights", "Mercy Drive", "Carver Shores"]
ZOOM_CHORO = 11
ZOOM_DOTS  = 14

_COLOR_HEX = {
    "darkred":   "#8b0000",
    "red":       "#cc0000",
    "orange":    "#e07800",
    "cadetblue": "#5f9ea0",
}

# JS template -- placeholders replaced at runtime, no nested Python/JS quote conflict
_JS_TMPL = (
    "\n<script>\n"
    "(function() {\n"
    "  var MAP        = __MAP__;\n"
    "  var ZOOM_CHORO = __ZOOM_CHORO__;\n"
    "  var ZOOM_DOTS  = __ZOOM_DOTS__;\n"
    "  var incidents  = __INCIDENTS__;\n"
    "  var allYears   = __ALL_YEARS__;\n"
    "  var yearNbhd   = __YEAR_NBHD__;\n"
    "  var choroData  = __CHORO_DATA__;\n"
    "  var selYears = new Set(allYears);\n"
    "  var clusterLyr = L.markerClusterGroup({\n"
    "    maxClusterRadius: 55, showCoverageOnHover: false,\n"
    "    iconCreateFunction: function(cl) {\n"
    "      var n = cl.getChildCount();\n"
    "      var sz = n > 200 ? 52 : n > 80 ? 42 : n > 20 ? 34 : 26;\n"
    "      var el = document.createElement('div');\n"
    "      el.textContent = String(n);\n"
    "      el.style.cssText = 'background:rgba(60,120,190,0.85);color:#fff;border-radius:50%;'\n"
    "          + 'width:' + sz + 'px;height:' + sz + 'px;display:flex;align-items:center;'\n"
    "          + 'justify-content:center;font-size:11px;font-weight:bold;'\n"
    "          + 'border:2px solid rgba(255,255,255,0.55);';\n"
    "      return L.divIcon({html: el.outerHTML, iconSize: [sz, sz], className: ''});\n"
    "    }\n"
    "  });\n"
    "  var dotLyr = L.featureGroup();\n"
    "  var choroLyr = null;\n"
    "  var RAMP = ['#fff5eb','#fee6ce','#fdd0a2','#fdae6b','#fd8d3c','#f16913','#d94801','#8c2d04'];\n"
    "  function choroColor(cnt, mx) {\n"
    "    if (!cnt) return '#f0f0f0';\n"
    "    return RAMP[Math.min(RAMP.length - 1, Math.floor((cnt / mx) * RAMP.length))];\n"
    "  }\n"
    "  function buildChoropleth() {\n"
    "    var nbhdCnt = {};\n"
    "    selYears.forEach(function(yr) {\n"
    "      var yc = yearNbhd[yr] || {};\n"
    "      Object.keys(yc).forEach(function(id) { nbhdCnt[id] = (nbhdCnt[id] || 0) + yc[id]; });\n"
    "    });\n"
    "    var mx = 1;\n"
    "    Object.keys(nbhdCnt).forEach(function(k) { if (nbhdCnt[k] > mx) mx = nbhdCnt[k]; });\n"
    "    if (choroLyr) { choroLyr.remove(); choroLyr = null; }\n"
    "    choroLyr = L.geoJSON(choroData, {\n"
    "      style: function(f) {\n"
    "        var cnt = nbhdCnt[f.properties.id] || 0;\n"
    "        return {fillColor: choroColor(cnt, mx), fillOpacity: cnt ? 0.72 : 0.08, color: '#555', weight: 0.8};\n"
    "      },\n"
    "      onEachFeature: function(f, lyr) {\n"
    "        var cnt = nbhdCnt[f.properties.id] || 0;\n"
    "        lyr.bindTooltip('<b>' + f.properties.n + '</b><br>' + cnt +\n"
    "          ' incident' + (cnt !== 1 ? 's' : ''), {sticky: true});\n"
    "      }\n"
    "    });\n"
    "  }\n"
    "  function buildIncidents() {\n"
    "    clusterLyr.clearLayers(); dotLyr.clearLayers();\n"
    "    incidents.forEach(function(inc) {\n"
    "      if (!selYears.has(inc.y)) return;\n"
    "      var opt = {radius:5, color:inc.c, fillColor:inc.c, fillOpacity:0.78, weight:1};\n"
    "      dotLyr.addLayer(L.circleMarker([inc.a,inc.o],opt).bindPopup(inc.p,{maxWidth:300}));\n"
    "      clusterLyr.addLayer(L.circleMarker([inc.a,inc.o],opt).bindPopup(inc.p,{maxWidth:300}));\n"
    "    });\n"
    "  }\n"
    "  function applyZoom() {\n"
    "    var z = MAP.getZoom();\n"
    "    if (MAP.hasLayer(clusterLyr)) MAP.removeLayer(clusterLyr);\n"
    "    if (MAP.hasLayer(dotLyr))     MAP.removeLayer(dotLyr);\n"
    "    if (choroLyr && MAP.hasLayer(choroLyr)) choroLyr.remove();\n"
    "    if      (z < ZOOM_CHORO) { MAP.addLayer(clusterLyr); }\n"
    "    else if (z < ZOOM_DOTS)  { if (choroLyr) choroLyr.addTo(MAP); }\n"
    "    else                     { MAP.addLayer(dotLyr); }\n"
    "  }\n"
    "  MAP.on('zoomend', applyZoom);\n"
    "  function refresh() {\n"
    "    buildIncidents(); buildChoropleth(); applyZoom();\n"
    "    var total = 0;\n"
    "    incidents.forEach(function(i) { if (selYears.has(i.y)) total++; });\n"
    "    var el = document.getElementById('map-sub');\n"
    "    if (el) {\n"
    "      var yrs = Array.from(selYears).sort();\n"
    "      el.textContent = total.toLocaleString() + ' incidents \\u2014 ' +\n"
    "        (yrs.length === allYears.length ? 'All years' :\n"
    "         yrs.length === 0 ? 'No years selected' : yrs.join(', '));\n"
    "    }\n"
    "  }\n"
    "  var panel = document.createElement('div');\n"
    "  panel.style.cssText = 'position:fixed;top:70px;right:10px;background:white;'\n"
    "    + 'padding:10px 14px;border-radius:6px;border:1px solid #aaa;z-index:1000;'\n"
    "    + 'font-family:Arial,sans-serif;font-size:13px;min-width:110px;max-height:80vh;overflow-y:auto;';\n"
    "  var hdr = document.createElement('div');\n"
    "  hdr.innerHTML = '<b>Years</b>&nbsp;'\n"
    "    + '<button id=\"yr-all\" style=\"font-size:10px;padding:1px 5px;cursor:pointer;margin:2px;\">All</button>'\n"
    "    + '<button id=\"yr-none\" style=\"font-size:10px;padding:1px 5px;cursor:pointer;\">None</button>';\n"
    "  panel.appendChild(hdr);\n"
    "  document.body.appendChild(panel);\n"
    "  allYears.forEach(function(yr) {\n"
    "    var row = document.createElement('div');\n"
    "    row.style.marginTop = '3px';\n"
    "    var cb = document.createElement('input');\n"
    "    cb.type = 'checkbox'; cb.checked = true; cb.id = 'cb' + yr;\n"
    "    cb.addEventListener('change', function() {\n"
    "      if (this.checked) selYears.add(yr); else selYears.delete(yr); refresh();\n"
    "    });\n"
    "    var lbl = document.createElement('label');\n"
    "    lbl.htmlFor = 'cb' + yr; lbl.textContent = ' ' + yr;\n"
    "    row.appendChild(cb); row.appendChild(lbl); panel.appendChild(row);\n"
    "  });\n"
    "  document.getElementById('yr-all').addEventListener('click', function() {\n"
    "    allYears.forEach(function(yr) {\n"
    "      selYears.add(yr); document.getElementById('cb' + yr).checked = true;\n"
    "    }); refresh();\n"
    "  });\n"
    "  document.getElementById('yr-none').addEventListener('click', function() {\n"
    "    selYears.clear();\n"
    "    allYears.forEach(function(yr) { document.getElementById('cb' + yr).checked = false; });\n"
    "    refresh();\n"
    "  });\n"
    "  refresh();\n"
    "})();\n"
    "</script>\n"
)


def map_interactive_all_years(df: pd.DataFrame) -> None:
    nbhd_gdf = gpd.read_file(str(config.NEIGHBORHOODS_DIR / "orlando_neighborhoods.geojson")).to_crs("EPSG:4326")
    kidz_gdf = gpd.read_file(str(config.NEIGHBORHOODS_DIR / "kidz_zones.geojson")).to_crs("EPSG:4326")

    inc_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    joined = gpd.sjoin(
        inc_gdf, nbhd_gdf[["NeighborhoodID", "geometry"]],
        how="left", predicate="within",
    ).drop(columns=["geometry", "index_right"], errors="ignore")

    year_nbhd: dict = {}
    for year, grp in joined.groupby("year"):
        counts = grp.dropna(subset=["NeighborhoodID"]).groupby("NeighborhoodID").size()
        year_nbhd[int(year)] = {int(k): int(v) for k, v in counts.items()}

    records = []
    for _, row in df.iterrows():
        color_key = _marker_color(row)
        date_str = row["date"].strftime("%B %d, %Y") if pd.notna(row["date"]) else "Unknown"
        popup = (
            f"<b>{date_str}</b><br>"
            f"{row.get('address', '')}<br>"
            f"Killed: {int(row['killed'])} | Injured: {int(row['injured'])}"
            + (" Mass Shooting" if row.get("mass") else "")
        )
        records.append({
            "a": float(row["lat"]),
            "o": float(row["lon"]),
            "c": _COLOR_HEX.get(color_key, "#888"),
            "y": int(row["year"]),
            "p": popup,
        })

    all_years = sorted(int(y) for y in df["year"].unique())

    nbhd_min = nbhd_gdf[["NeighborhoodID", "NeighborhoodName", "geometry"]].copy()
    choro_json = _json.loads(nbhd_min.to_json())
    for feat in choro_json["features"]:
        p = feat["properties"]
        feat["properties"] = {"id": p["NeighborhoodID"], "n": p["NeighborhoodName"]}

    m = _base_map()
    MAP = m.get_name()

    m.get_root().header.add_child(folium.Element(
        '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/'
        'leaflet.markercluster/1.1.0/MarkerCluster.css"/>\n'
        '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/'
        'leaflet.markercluster/1.1.0/MarkerCluster.Default.css"/>\n'
        '<script src="https://cdnjs.cloudflare.com/ajax/libs/'
        'leaflet.markercluster/1.1.0/leaflet.markercluster.js"></script>'
    ))

    preferred_gdf = nbhd_min[nbhd_min["NeighborhoodName"].isin(PREFERRED_NAMES)]

    folium.GeoJson(
        nbhd_min.to_json(), name="Neighborhoods (all 125)", show=False,
        style_function=lambda f: {"color": "#888", "weight": 1, "fillColor": "#ccc", "fillOpacity": 0.05},
        tooltip=folium.GeoJsonTooltip(["NeighborhoodName"], aliases=["Neighborhood:"]),
    ).add_to(m)

    folium.GeoJson(
        preferred_gdf.to_json(), name="Preferred Neighborhoods (4)", show=False,
        style_function=lambda f: {"color": "#e76f51", "weight": 2.5, "fillColor": "#e76f51", "fillOpacity": 0.18},
        tooltip=folium.GeoJsonTooltip(["NeighborhoodName"], aliases=["Neighborhood:"]),
    ).add_to(m)

    folium.GeoJson(
        kidz_gdf[["NAME", "ADDRESS", "geometry"]].to_json(),
        name="Kidz Zones (121)", show=False,
        style_function=lambda f: {"color": "#2ca25f", "weight": 1.5, "fillColor": "#2ca25f", "fillOpacity": 0.15},
        tooltip=folium.GeoJsonTooltip(["NAME", "ADDRESS"], aliases=["Zone:", "Address:"]),
    ).add_to(m)

    folium.LayerControl(collapsed=True).add_to(m)

    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);'
        f'background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;'
        f'z-index:1001;font-family:Arial,sans-serif;font-size:14px;text-align:center;white-space:nowrap;">'
        f'<b>Orlando Gun Violence - All Years</b><br>'
        f'<span id="map-sub" style="font-size:12px;font-weight:normal;">'
        f'{len(df):,} incidents 2014-2025</span></div>'
        f'<div style="position:fixed;bottom:30px;left:30px;background:white;'
        f'padding:10px 14px;border-radius:6px;border:1px solid #aaa;'
        f'z-index:1000;font-family:Arial,sans-serif;font-size:13px;">'
        f'<b>Incident Type</b><br>'
        f'<div><span style="color:#8b0000;font-size:16px;">&#9679;</span> Mass shooting</div>'
        f'<div><span style="color:#cc0000;font-size:16px;">&#9679;</span> Fatal</div>'
        f'<div><span style="color:#e07800;font-size:16px;">&#9679;</span> Multiple injured</div>'
        f'<div><span style="color:#5f9ea0;font-size:16px;">&#9679;</span> Injury only</div>'
        f'<hr style="margin:5px 0;">'
        f'<div style="font-size:11px;color:#555;line-height:1.5;">'
        f'Far out: clusters<br>Mid zoom: density<br>Close: dots</div></div>'
    ))

    js = (
        _JS_TMPL
        .replace("__MAP__",        MAP)
        .replace("__ZOOM_CHORO__", str(ZOOM_CHORO))
        .replace("__ZOOM_DOTS__",  str(ZOOM_DOTS))
        .replace("__INCIDENTS__",  _json.dumps(records))
        .replace("__ALL_YEARS__",  _json.dumps(all_years))
        .replace("__YEAR_NBHD__",  _json.dumps(year_nbhd))
        .replace("__CHORO_DATA__", _json.dumps(choro_json))
    )
    m.get_root().html.add_child(folium.Element(js))

    out = config.OUTPUT_MAPS / "orlando_interactive_full.html"
    m.save(str(out))
    print(f"  Saved: {out.name}")


In [8]:
# Run this cell to generate orlando_interactive_full.html
df_all = load_data()
map_interactive_all_years(df_all)
print("Done - open output/maps/orlando_interactive_full.html in a browser.")


Loaded 1,742 mappable incidents.
  Saved: orlando_interactive_full.html
Done - open output/maps/orlando_interactive_full.html in a browser.
